In [1]:
import numpy as np
import pandas as pd
from collections import Counter

In [ ]:
df = pd.read_csv("../data/ble_data_labeled_cleaned.csv")
df = df.iloc[:2000]
df.head()

,user_id,timestamp,mac_address,RSSI,power,location
0,90,2023-04-10 14:21:46.003,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
1,90,2023-04-10 14:21:46.008,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
2,90,2023-04-10 14:21:46.012,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
3,90,2023-04-10 14:21:46.018,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
4,90,2023-04-10 14:21:46.024,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen


In [10]:
def window_ble_data(df, window_size_s=2.0, step_size_s=1.0):
    df = df.sort_values("timestamp").copy()

    if not np.issubdtype(df["timestamp"].dtype, np.datetime64):
        df["timestamp"] = pd.to_datetime(df["timestamp"])

    start = df["timestamp"].min()
    end   = df["timestamp"].max()

    windows = []
    t = start

    while t + pd.Timedelta(seconds=window_size_s) <= end:
        w = df[
            (df["timestamp"] >= t) &
            (df["timestamp"] <  t + pd.Timedelta(seconds=window_size_s))
        ]
        if not w.empty:
            windows.append(w)
        t += pd.Timedelta(seconds=step_size_s)

    return windows

In [6]:
def normalize_rssi_per_window(window_df):
    rssi = window_df["RSSI"].values
    if len(rssi) < 2:
        window_df["rssi_norm"] = 0.0
        return window_df

    window_df["rssi_norm"] = (
        rssi - np.mean(rssi)
    ) / (np.std(rssi) + 1e-6)

    return window_df

In [ ]:
def extract_features_from_window(
    window_df,
    top_k=5
):
    features = {}

    # Normalize RSSI (per window, per device)
    window_df = normalize_rssi_per_window(window_df)

    grouped = window_df.groupby("mac_address")
    beacon_stats = []

    for mac, g in grouped:
        mean_rssi = g["RSSI"].mean()
        std_rssi  = g["RSSI"].std()
        mean_norm = g["rssi_norm"].mean()
        count     = len(g)

        beacon_stats.append({
            "mac": mac,
            "mean_rssi": mean_rssi,
            "std_rssi": std_rssi,
            "mean_norm": mean_norm,
            "count": count,
        })

    if not beacon_stats:
        return None

    # Sort by strongest signal
    beacon_stats.sort(key=lambda x: x["mean_rssi"], reverse=True)

    # ---- Global features ----
    features["num_beacons_seen"] = len(beacon_stats)
    features["strongest_rssi"] = beacon_stats[0]["mean_rssi"]

    if len(beacon_stats) > 1:
        features["rssi_gap_1_2"] = (
            beacon_stats[0]["mean_rssi"] -
            beacon_stats[1]["mean_rssi"]
        )
    else:
        features["rssi_gap_1_2"] = 0.0

    # ---- Top-K relative beacon features ----
    for i, b in enumerate(beacon_stats[:top_k]):
        features[f"b{i}_rssi_norm"] = b["mean_norm"]
        features[f"b{i}_rssi_std"]  = b["std_rssi"]
        features[f"b{i}_count"]     = b["count"]


    # ---- Label (majority vote in window) ----
    features["location"] = window_df["location"].mode()[0]

    return features

In [8]:
def extract_dataset_features(
    df,
    window_size_s=2.0,
    step_size_s=1.0
):
    windows = window_ble_data(df, window_size_s, step_size_s)
    rows = []

    for w in windows:
        feats = extract_features_from_window(
            w
        )
        if feats is not None:
            rows.append(feats)

    return pd.DataFrame(rows)

In [11]:
features_df = extract_dataset_features(
    df,
    window_size_s=2.0,
    step_size_s=1.0
)
features_df.head()

KeyboardInterrupt: 